# Lakebase check (2/2): the decisions after write-back

Runs after the `write_back` task (`src/lakebase/run_write_back.py`, the same file the
`bigbox_refresh` job runs). Every evidence decision id from the `seed` task must be in Delta
`gold.reorder_decisions` with the same values as in Postgres `app.reorder_decisions`.
The job's last task runs the write-back again, which must insert 0 rows.

In [ ]:
import json

import pandas as pd
from pyspark.sql import functions as F

from bigbox_lakebase import connect

dbutils.widgets.text("catalog", "bigbox_fe_bar")
CAT = dbutils.widgets.get("catalog")
ids = dbutils.jobs.taskValues.get(taskKey="seed", key="decision_ids", debugValue=[])
print(f"{len(ids)} decision ids from the seed task")

20 decision ids from the seed task


In [ ]:
COLS = ["store_id", "sku_id", "recommended_qty", "final_qty", "action", "reason", "decided_by"]
conn = connect()
cur = conn.execute(
    "SELECT decision_id::text, " + ", ".join(COLS) + ", decided_at"
    " FROM app.reorder_decisions WHERE decision_id = ANY(%s::uuid[])",
    (ids,),
)
pg = pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description])
pg_total = conn.execute("SELECT count(*) FROM app.reorder_decisions").fetchone()[0]
conn.close()
delta = (
    spark.read.table(f"{CAT}.gold.reorder_decisions")
    .where(F.col("decision_id").isin(ids))
    .select("decision_id", *COLS, "region", "decided_at", "written_back_at")
    .toPandas()
)
delta_total = spark.read.table(f"{CAT}.gold.reorder_decisions").count()
m = pg.merge(delta, on="decision_id", how="outer", suffixes=("_pg", "_delta"), indicator=True)
mismatch = {
    c: int(
        ((m[f"{c}_pg"].astype(str) != m[f"{c}_delta"].astype(str)) & (m["_merge"] == "both")).sum()
    )
    for c in COLS
}
written = pd.to_datetime(m["written_back_at"], utc=True)
lag = written - pd.to_datetime(m["decided_at_pg"], utc=True)
result = {
    "seeded_ids": len(ids),
    "in_postgres": int(pg["decision_id"].isin(ids).sum()),
    "in_delta": int(delta["decision_id"].isin(ids).sum()),
    "matching_ids": int((m["_merge"] == "both").sum()),
    "column_mismatches": mismatch,
    "region_missing": int(delta["region"].isna().sum()),
    "postgres_total": pg_total,
    "delta_total": delta_total,
    "decided_to_written_back_s_max": round(lag.dt.total_seconds().max(), 1),
}
print(json.dumps(result, indent=2))
assert result["matching_ids"] == len(ids) == result["in_delta"] and not any(mismatch.values())

{
  "seeded_ids": 20,
  "in_postgres": 20,
  "in_delta": 20,
  "matching_ids": 20,
  "column_mismatches": {
    "store_id": 0,
    "sku_id": 0,
    "recommended_qty": 0,
    "final_qty": 0,
    "action": 0,
    "reason": 0,
    "decided_by": 0
  },
  "region_missing": 0,
  "postgres_total": 20,
  "delta_total": 20,
  "decided_to_written_back_s_max": 87.4
}


In [ ]:
display(
    spark.read.table(f"{CAT}.gold.reorder_decisions")
    .where(F.col("decision_id").isin(ids))
    .groupBy("action")
    .agg(
        F.count("*").alias("decisions"),
        F.sum("recommended_qty").alias("recommended_units"),
        F.sum("final_qty").alias("final_units"),
        F.countDistinct("region").alias("regions"),
    )
    .orderBy("action")
)
dbutils.notebook.exit(json.dumps(result))  # read by scripts/evidence_lakebase.py

{"seeded_ids": 20, "in_postgres": 20, "in_delta": 20, "matching_ids": 20, "column_mismatches": {"store_id": 0, "sku_id": 0, "recommended_qty": 0, "final_qty": 0, "action": 0, "reason": 0, "decided_by": 0}, "region_missing": 0, "postgres_total": 20, "delta_total": 20, "decided_to_written_back_s_max": 87.4}